In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import math
from glob import glob
import re
import os
import re
import math
from sklearn.model_selection import train_test_split
import sentencepiece as spm

In [3]:
TRAIN_PATH = '/kaggle/input/datasets/uk2400/nlpass3/hindi_corpus/hindi_corpus/train'
SENTIMENT  = '/kaggle/input/datasets/uk2400/nlpass3/text_classification_dataset/text_classification_dataset/train.csv'
block_size = 512 

In [4]:
batch_size  = 32
block_size  = 512
vocab_size  = 5000
n_embd      = 128      
n_head      = 4
n_layer     = 4
dropout     = 0.1     
device      = 'cuda' if torch.cuda.is_available() else 'cpu'
max_iter    = 240_000
eval_int    = 2_000   
eval_iter   = 200

lr_max      = 3e-4
lr_min      = 1e-4
warmup_steps = 2_000

torch.manual_seed(2023566)
print("Device:", device)
 

Device: cuda


In [5]:
def is_hindi(text):
    return bool(re.search(r'[\u0900-\u097F]', text))
 
def load_split(data_dir, val_size=0.1):
    all_lines = []
    for f_path in glob(os.path.join(data_dir, "*.txt")):
        with open(f_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and is_hindi(line):
                    all_lines.append(line)
    print(f"Total Hindi lines loaded: {len(all_lines)}")
    train_data, val_data = train_test_split(all_lines, test_size=val_size, random_state=42)
    print(f"Train: {len(train_data)} | Val: {len(val_data)}")
    return train_data, val_data
 
def train_spm(train_list, vocab_size):
    spm.SentencePieceTrainer.train(
        sentence_iterator=iter(train_list),
        model_prefix='hindi_bpe',
        vocab_size=vocab_size,
        model_type='bpe',
        normalization_rule_name='nfkc',
        character_coverage=1.0,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3
    )
    sp = spm.SentencePieceProcessor()
    sp.load('hindi_bpe.model')
    print(f"Tokenizer trained. Vocab size: {sp.get_piece_size()}")
    return sp
 
def encode(sp_processor, text):
    return sp_processor.encode_as_ids(text)
 
def decode(sp_processor, ids):
    return sp_processor.decode_ids(ids)
 
def LM_pair(token_sequences):
    pairs = []
    for seq in token_sequences:
        if len(seq) >= 2:
            pairs.append((seq[:-1], seq[1:]))
    return pairs

In [6]:
langtrain, langval = load_split(TRAIN_PATH, 0.1)
sp_processor       = train_spm(langtrain, vocab_size)
 
train_tokens   = [encode(sp_processor, line) for line in langtrain]
val_tokens     = [encode(sp_processor, line) for line in langval]
 
lm_train_pairs = LM_pair(train_tokens)
lm_val_pairs   = LM_pair(val_tokens)
 
print(f"Train pairs: {len(lm_train_pairs)} | Val pairs: {len(lm_val_pairs)}")
 

Total Hindi lines loaded: 210024
Train: 189021 | Val: 21003
Tokenizer trained. Vocab size: 5000


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input_format: 
  model_prefix: hindi_bpe
  model_type: BPE
  vocab_size: 5000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noise_level: 0


Train pairs: 188560 | Val pairs: 20951


In [7]:
PAD_IDX = sp_processor.pad_id()  
 
class NLPDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        x, y = self.pairs[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)
 
def lm_collate(batch):
    xs = [item[0] for item in batch]
    ys = [item[1] for item in batch]
    x_padded = pad_sequence(xs, batch_first=True, padding_value=PAD_IDX)
    y_padded = pad_sequence(ys, batch_first=True, padding_value=-100) 
    if x_padded.shape[1] > block_size:
        x_padded = x_padded[:, :block_size]
        y_padded = y_padded[:, :block_size]
    return x_padded, y_padded
 
lm_train_loader = DataLoader(NLPDataset(lm_train_pairs), batch_size=batch_size,
                              shuffle=True,  collate_fn=lm_collate)
lm_val_loader   = DataLoader(NLPDataset(lm_val_pairs),   batch_size=batch_size,
                              shuffle=False, collate_fn=lm_collate)
 
def get_lr(step):
    if step < warmup_steps:
        return lr_max * step / warmup_steps
    progress = (step - warmup_steps) / (max_iter - warmup_steps)
    cosine   = 0.5 * (1 + math.cos(math.pi * progress))
    return lr_min + (lr_max - lr_min) * cosine

xb, yb = next(iter(lm_train_loader))
print(f"Batch shapes — x: {xb.shape}, y: {yb.shape}")

Batch shapes — x: torch.Size([32, 340]), y: torch.Size([32, 340])


In [8]:

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.val   = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
 
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        sim = (q @ k.transpose(-2, -1)) * (k.size(-1) ** -0.5)
        sim = sim.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        sim = F.softmax(sim, dim=-1)
        sim = self.dropout(sim)
        v   = self.val(x)
        return sim @ v
 
 
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
 
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))
 
 
class FFN(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
 
    def forward(self, x):
        return self.net(x)
 
 
class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size  = n_embd // n_head
        self.sa    = MultiHeadAttention(n_head, head_size)
        self.ffn   = FFN(n_embd)
        self.ln1   = nn.LayerNorm(n_embd)
        self.ln2   = nn.LayerNorm(n_embd)
 
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x
 
 
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.Sequential(*[TransformerBlock(n_embd, n_head)
                                        for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
 
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.tok_emb(idx)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device)).unsqueeze(0)
        x = self.drop(tok_emb + pos_emb)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                           # (B, T, vocab_size)
 
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits_2d  = logits.reshape(B * T, C)
            targets_1d = targets.reshape(B * T)
            
            loss = F.cross_entropy(logits_2d, targets_1d, ignore_index=-100)
 
        return logits, loss
 
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs  = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            if idx_next.item() in [0, 1, 2, 3]:  # stop on any special token
                break
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [9]:
model = GPTLanguageModel().to(device)
 
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
 
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
 
optimizer = torch.optim.AdamW(model.parameters(), lr=lr_max, weight_decay=0.01)

@torch.no_grad()
def est_loss():
    out = {}
    model.eval()
    for split, loader in [('train', lm_train_loader), ('val', lm_val_loader)]:
        losses = []
        for k, (X, Y) in enumerate(loader):
            if k >= eval_iter:
                break
            X, Y = X.to(device), Y.to(device)
            _, loss = model(X, Y)
            losses.append(loss.mean().item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

Using 2 GPUs
Parameters: 2,142,344


In [ ]:
track_steps, track_train, track_val = [], [], []
data_iter = iter(lm_train_loader)
best_val  = float('inf')
 
print(f"Training for {max_iter:,} steps")
 
for step in range(max_iter):
 
    
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg['lr'] = lr
 
    if step % eval_int == 0 or step == max_iter - 1:
        losses = est_loss()
        perp   = math.exp(losses['val'])
        print(f"step {step:6d} | lr {lr:.2e} | "
              f"train {losses['train']:.4f} | val {losses['val']:.4f} | perp {perp:.1f}")
        track_steps.append(step)
        track_train.append(losses['train'])
        track_val.append(losses['val'])
 
        if losses['val'] < best_val:
            best_val = losses['val']
            m = model.module if isinstance(model, nn.DataParallel) else model
            torch.save(m.state_dict(), 'hindi_lm_best.pth')
            print(f"  --> new best val loss {best_val:.4f}, saved checkpoint")
 
    if step > 0 and step % 20_000 == 0:
        m = model.module if isinstance(model, nn.DataParallel) else model
        torch.save(m.state_dict(), f'hindi_lm_step{step}.pth')
 
    try:
        xb, yb = next(data_iter)
    except StopIteration:
        data_iter = iter(lm_train_loader)
        xb, yb = next(data_iter)
 
    xb, yb = xb.to(device), yb.to(device)
    logits, loss = model(xb, yb)
    loss = loss.mean()                   
 
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  
    optimizer.step()
 
print(f"\nTraining complete. Best val loss: {best_val:.4f}")

In [ ]:
model = GPTLanguageModel().to(device)
# model.load_state_dict(torch.load('hindi_lm_best.pth', map_location=device))
model.load_state_dict(torch.load('/kaggle/input/models/uk2400/nlpbest/pytorch/default/1/hindi_lm_best (1).pth', map_location=device))
model.eval()
print("Model loaded")

In [ ]:
model.eval()
with torch.no_grad():
    final = est_loss()
    val_perplexity = math.exp(final['val'])
    print(f"Final val loss:       {final['val']:.4f}")
    print(f"Final val perplexity: {val_perplexity:.2f}")
 

print("\n Hindi Tokens")
bos_id  = sp_processor.bos_id()          
context = torch.tensor([[bos_id]], dtype=torch.long, device=device)
 
m = model.module if isinstance(model, nn.DataParallel) else model
generated_ids  = m.generate(context, max_new_tokens=50, temperature=0.8)
generated_text = decode(sp_processor, generated_ids[0].tolist())
print(generated_text)
 
with open('task2_output.txt', 'w', encoding='utf-8') as f:
    f.write(f"Final val loss:       {final['val']:.4f}\n")
    f.write(f"Final val perplexity: {val_perplexity:.2f}\n\n")
    f.write("Generated Hindi Text (100 tokens):\n")
    f.write(generated_text + "\n")

print("task2_output.txt saved!")

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.plot(track_steps, track_train, label='train loss')
    plt.plot(track_steps, track_val,   label='val loss')
    plt.xlabel('step')
    plt.ylabel('cross-entropy loss')
    plt.title('Hindi language model — training curve')
    plt.legend()
    plt.tight_layout()
    plt.savefig('loss_curve.png', dpi=150)
    print("Loss curve saved to loss_curve.png")
except ImportError:
    pass

In [ ]:

CLASSIFICATION_PATH = SENTIMENT  
MODEL_CHECKPOINT = 'hindi_lm_best.pth'
cls_batch_size      = 16
cls_epochs          = 40
num_classes         = 3
lr_last_blocks      = 5e-5
lr_head             = 5e-4
early_stop_patience = 8

block_size = 512
vocab_size = 5000
n_embd     = 128
n_head     = 4
n_layer    = 4
dropout    = 0.1
device     = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(42)
print("Device:", device)

def load_classification_data(path, val_size=0.1):
    df = pd.read_csv(path)
    print(f"Loaded {len(df)} rows.")
    texts  = df['text'].astype(str).tolist()
    labels = df['experience'].astype(int).tolist()
    assert set(labels).issubset({0, 1, 2})
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels, test_size=val_size, random_state=42, stratify=labels
    )
    print(f"Train: {len(train_texts)} | Val: {len(val_texts)}")
    print(f"Class counts — 0:{labels.count(0)}  1:{labels.count(1)}  2:{labels.count(2)}")
    return train_texts, val_texts, train_labels, val_labels

train_texts, val_texts, train_labels, val_labels = load_classification_data(
    CLASSIFICATION_PATH
)


class ClassificationDataset(Dataset):
    def __init__(self, texts, labels, sp_processor):
        self.samples = []
        for text, label in zip(texts, labels):
            ids = sp_processor.encode_as_ids(text)
            if len(ids) == 0:
                continue
            self.samples.append((ids[:block_size], label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids, label = self.samples[idx]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

def cls_collate(batch):
    xs      = [item[0] for item in batch]
    labels  = torch.stack([item[1] for item in batch])
    lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)
    return pad_sequence(xs, batch_first=True, padding_value=0), labels, lengths

train_cls_loader = DataLoader(
    ClassificationDataset(train_texts, train_labels, sp_processor),
    batch_size=cls_batch_size, shuffle=True, collate_fn=cls_collate)
val_cls_loader = DataLoader(
    ClassificationDataset(val_texts, val_labels, sp_processor),
    batch_size=cls_batch_size, shuffle=False, collate_fn=cls_collate)

class GPTClassifier(nn.Module):
    def __init__(self, pretrained_path, num_classes):
        super().__init__()

        self.backbone = GPTLanguageModel()
        self.backbone.load_state_dict(torch.load(pretrained_path, map_location=device))
        print("Backbone loaded.")

   
        for param in self.backbone.parameters():
            param.requires_grad = False
        for block in self.backbone.blocks[-2:]:
            for param in block.parameters():
                param.requires_grad = True
        for param in self.backbone.ln_f.parameters():
            param.requires_grad = True

        frozen   = sum(p.numel() for p in self.backbone.parameters() if not p.requires_grad)
        unfrozen = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        print(f"Backbone — frozen: {frozen:,} | unfrozen: {unfrozen:,}")

        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(n_embd, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, idx, lengths):
        B, T = idx.shape

        tok_emb = self.backbone.tok_emb(idx)
        pos_emb = self.backbone.pos_emb(torch.arange(T, device=idx.device)).unsqueeze(0)
        x = self.backbone.drop(tok_emb + pos_emb)
        x = self.backbone.blocks(x)
        x = self.backbone.ln_f(x)
        # x: (B, T, n_embd)

        
        batch_indices = torch.arange(B, device=idx.device)
        pooled = x[batch_indices, lengths - 1]

        return self.classifier(pooled)


classifier = GPTClassifier(MODEL_CHECKPOINT, num_classes).to(device)

trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
total     = sum(p.numel() for p in classifier.parameters())
print(f"Total trainable: {trainable:,} / {total:,}")

optimizer_cls = torch.optim.AdamW([
    {'params': [p for p in classifier.backbone.parameters() if p.requires_grad],
     'lr': lr_last_blocks},
    {'params': list(classifier.classifier.parameters()),
     'lr': lr_head},
], weight_decay=0.05)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_cls, T_max=cls_epochs, eta_min=1e-6
)

def evaluate_classifier(model, loader):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb, lengths in loader:
            xb, yb, lengths = xb.to(device), yb.to(device), lengths.to(device)
            logits = model(xb, lengths)
            loss   = F.cross_entropy(logits, yb)
            preds  = logits.argmax(dim=-1)
            correct    += (preds == yb).sum().item()
            total      += yb.size(0)
            total_loss += loss.item() * yb.size(0)
    return total_loss / total, correct / total



best_val_acc      = 0.0
epochs_no_improve = 0

train_accs, val_accs, train_losses, val_losses = [], [], [], []  


for epoch in range(1, cls_epochs + 1):
    classifier.train()
    train_loss_total, train_correct, train_total = 0.0, 0, 0

    for xb, yb, lengths in train_cls_loader:
        xb, yb, lengths = xb.to(device), yb.to(device), lengths.to(device)
        logits = classifier(xb, lengths)
        loss   = F.cross_entropy(logits, yb)

        optimizer_cls.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
        optimizer_cls.step()

        preds             = logits.argmax(dim=-1)
        train_correct    += (preds == yb).sum().item()
        train_total      += yb.size(0)
        train_loss_total += loss.item() * yb.size(0)

    scheduler.step()

    train_acc  = train_correct / train_total
    train_loss = train_loss_total / train_total
    val_loss, val_acc = evaluate_classifier(classifier, val_cls_loader)
    train_accs.append(train_acc)       
    val_accs.append(val_acc)           
    train_losses.append(train_loss)    
    val_losses.append(val_loss)

    
    if val_acc > best_val_acc:
        best_val_acc      = val_acc
        epochs_no_improve = 0
        torch.save(classifier.state_dict(), 'hindi_classifier_best.pth')
        marker = " --> best saved"
    else:
        epochs_no_improve += 1
        marker = f" (no improvement {epochs_no_improve}/{early_stop_patience})"

    print(f"Epoch {epoch:2d}/{cls_epochs} | "
          f"train loss {train_loss:.4f} | train acc {train_acc:.4f} | "
          f"val loss {val_loss:.4f} | val acc {val_acc:.4f}{marker}")

    if epochs_no_improve >= early_stop_patience:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest validation accuracy: {best_val_acc*100:.2f}%")

classifier.load_state_dict(torch.load('hindi_classifier_best.pth', map_location=device))

print("\n--- Final Evaluation (best checkpoint) ---")
_, final_acc = evaluate_classifier(classifier, val_cls_loader)
print(f"Final validation accuracy: {final_acc*100:.2f}%")

classifier.eval()
class_correct = [0, 0, 0]
class_total   = [0, 0, 0]

with torch.no_grad():
    for xb, yb, lengths in val_cls_loader:
        xb, yb, lengths = xb.to(device), yb.to(device), lengths.to(device)
        preds = classifier(xb, lengths).argmax(dim=-1)
        for label, pred in zip(yb.tolist(), preds.tolist()):
            class_total[label]   += 1
            class_correct[label] += int(pred == label)

print("\nPer-class accuracy:")
for i, name in enumerate(['Negative', 'Neutral', 'Positive']):
    if class_total[i] > 0:
        acc = class_correct[i] / class_total[i]
        print(f"  {name:8s}: {acc*100:.2f}%  ({class_correct[i]}/{class_total[i]})")

with open('task3_output.txt', 'w', encoding='utf-8') as f:
    f.write(f"Final validation accuracy: {final_acc*100:.2f}%\n\n")
    f.write("Per-class accuracy:\n")
    for i, name in enumerate(['Negative', 'Neutral', 'Positive']):
        if class_total[i] > 0:
            acc = class_correct[i] / class_total[i]
            f.write(f"  {name:8s}: {acc*100:.2f}%  ({class_correct[i]}/{class_total[i]})\n")

    f.write("\n5 Correctly Predicted Reviews:\n")
    f.write("-" * 40 + "\n")
    class_names = ['Negative', 'Neutral', 'Positive']
    count = 0
    classifier.eval()
    with torch.no_grad():
        for xb, yb, lengths in val_cls_loader:
            xb, yb, lengths = xb.to(device), yb.to(device), lengths.to(device)
            preds = classifier(xb, lengths).argmax(dim=-1)
            for i in range(len(yb)):
                if count >= 5:
                    break
                if preds[i].item() == yb[i].item():
                    tokens = xb[i][:lengths[i]].tolist()
                    text   = decode(sp_processor, tokens)
                    label  = class_names[yb[i].item()]
                    f.write(f"[{count+1}] True: {label} | Predicted: {label}\n")
                    f.write(f"    {text}\n\n")
                    count += 1
            if count >= 5:
                break

print("task3_output.txt saved!")

In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(train_accs) + 1), train_accs, label='train acc')
plt.plot(range(1, len(val_accs) + 1),   val_accs,   label='val acc')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.title('Classifier — accuracy curve')
plt.legend()
plt.tight_layout()
plt.savefig('task3_accuracy_curve.png', dpi=150)
print("task3_accuracy_curve.png saved!")

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='train loss')
plt.plot(range(1, len(val_losses) + 1),   val_losses,   label='val loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Classifier — loss curve')
plt.legend()
plt.tight_layout()
plt.savefig('task3_loss_curve.png', dpi=150)
print("task3_loss_curve.png saved!")